# DE5 Spark + Lakehouse 스타터 노트북

PySpark 3.5.6 (번들) 로컬 세션으로 Paimon / Iceberg 카탈로그를 탐색합니다.

- `paimon_lake.bronze.*` : Flink가 적재한 실시간/current-state 테이블 (`s3://paimon/warehouse`)
- `iceberg_lake.analytics.*` : Spark batch가 만든 분석 테이블 (Iceberg REST)

Iceberg/Paimon jar는 `spark-ivy-cache` 볼륨에서 재사용되므로 첫 세션 생성이 빠릅니다.

In [ ]:
from pyspark.sql import SparkSession

PAIMON_VERSION = "1.4.1"
ICEBERG_VERSION = "1.11.0"
PACKAGES = ",".join([
    f"org.apache.paimon:paimon-spark-3.5_2.12:{PAIMON_VERSION}",
    f"org.apache.paimon:paimon-s3:{PAIMON_VERSION}",
    f"org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:{ICEBERG_VERSION}",
    f"org.apache.iceberg:iceberg-aws-bundle:{ICEBERG_VERSION}",
])

spark = (
    SparkSession.builder.appName("de5-jupyter")
    .config("spark.jars.packages", PACKAGES)
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
            "org.apache.paimon.spark.extensions.PaimonSparkSessionExtensions")
    # --- Paimon (s3://paimon/warehouse) ---
    .config("spark.sql.catalog.paimon_lake", "org.apache.paimon.spark.SparkCatalog")
    .config("spark.sql.catalog.paimon_lake.warehouse", "s3://paimon/warehouse")
    .config("spark.sql.catalog.paimon_lake.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.paimon_lake.s3.access-key", "minioadmin")
    .config("spark.sql.catalog.paimon_lake.s3.secret-key", "minioadmin")
    .config("spark.sql.catalog.paimon_lake.s3.path.style.access", "true")
    # --- Iceberg (REST) ---
    .config("spark.sql.catalog.iceberg_lake", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.iceberg_lake.type", "rest")
    .config("spark.sql.catalog.iceberg_lake.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.iceberg_lake.warehouse", "s3://warehouse/")
    .config("spark.sql.catalog.iceberg_lake.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.iceberg_lake.s3.endpoint", "http://minio:9000")
    .config("spark.sql.catalog.iceberg_lake.s3.path-style-access", "true")
    .config("spark.sql.catalog.iceberg_lake.client.region", "us-east-1")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

## 1. Paimon bronze 테이블 (실시간/current-state)

In [ ]:
spark.sql("SHOW TABLES IN paimon_lake.bronze").show(truncate=False)
for t in ["ux_events_bronze", "review_current", "order_current"]:
    n = spark.sql(f"SELECT COUNT(*) c FROM paimon_lake.bronze.{t}").collect()[0]["c"]
    print(f"{t:20s} {n:>8,}")

In [ ]:
# upsert(current-state) 증거: order 마지막 상태 분포
spark.sql("""
  SELECT last_event_type, COUNT(*) AS cnt
  FROM paimon_lake.bronze.order_current
  GROUP BY last_event_type ORDER BY cnt DESC
""").show()

## 2. Snapshot 기준점 찾기: Paimon vs Iceberg

실무에서 snapshot은 '멋있는 time travel 기능'이 아니라 장애/배포/데이터 이상이 있었을 때 **어느 시점까지 정상으로 볼 수 있는가**를 찾는 운영 기준점입니다.

- Paimon: `table$snapshots`에서 `commit_time`, `commit_kind`, `commit_user`, `total_record_count`를 봅니다.
- Iceberg: `table.snapshots`에서 `committed_at`, `snapshot_id`, `operation`을 봅니다.
- 기준 snapshot을 찾은 뒤, Paimon은 `VERSION AS OF` / `TIMESTAMP AS OF`로 과거 상태를 읽어봅니다.



In [ ]:
# Paimon snapshot 이력: Flink streaming job이 commit을 만들고 있는지, APPEND/COMPACT가 보이는지 확인합니다.
# 주의: Spark/Paimon connector에서 `$snapshots`의 commit_time predicate pushdown이 깨지는 경우가 있습니다.
# 그래서 system table은 먼저 넓게 읽고, 특정 시각 필터는 Python에서 적용합니다.
paimon_snapshots = spark.sql("""
SELECT
  snapshot_id,
  commit_kind,
  commit_time,
  commit_user,
  total_record_count,
  delta_record_count
FROM paimon_lake.bronze.`review_current$snapshots`
ORDER BY snapshot_id DESC
LIMIT 50
""")
paimon_snapshots.show(truncate=False)

# 특정 장애/배포 시점 직전의 정상 snapshot 후보를 찾는 패턴입니다.
# 수업 중에는 현재 시각 기준으로 실행하고, 실제 장애 분석에서는 incident_time을 바꿔서 봅니다.
incident_time = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]
print(f"incident_time candidate = {incident_time}")

snapshot_rows = paimon_snapshots.collect()
normal_candidates = [
    r for r in snapshot_rows
    if r["commit_time"] is not None and r["commit_time"] <= incident_time
]
normal_candidates = sorted(normal_candidates, key=lambda r: r["commit_time"], reverse=True)[:5]

print("problem 직전 정상 snapshot 후보")
for r in normal_candidates:
    print(
        f"snapshot_id={r['snapshot_id']}, "
        f"commit_kind={r['commit_kind']}, "
        f"commit_time={r['commit_time']}, "
        f"total_record_count={r['total_record_count']}"
    )

# Time travel: 위에서 찾은 snapshot_id로 과거 상태를 직접 읽습니다.
# 후보가 없으면 snapshot_id=1을 사용합니다.
snapshot_id = normal_candidates[0]["snapshot_id"] if normal_candidates else 1
print(f"time travel snapshot_id = {snapshot_id}")

spark.sql(f"""
SELECT COUNT(*) AS review_current_rows_at_snapshot
FROM paimon_lake.bronze.review_current VERSION AS OF {snapshot_id}
""").show(truncate=False)

# timestamp 기준 time travel도 가능합니다. commit_time 중 하나를 복사해서 넣으면 됩니다.
# 단, system table 조회와 달리 time travel 자체에 쓰는 timestamp입니다.
# spark.sql("""
# SELECT COUNT(*)
# FROM paimon_lake.bronze.review_current TIMESTAMP AS OF '2026-06-07 21:00:00.000'
# """).show()



## 3. Iceberg-compatible metadata 확인

우리 Paimon 테이블은 `metadata.iceberg.storage = rest-catalog` 설정으로 Iceberg-compatible metadata도 생성합니다. 이건 데이터를 복사하는 것이 아니라, Iceberg reader가 같은 테이블을 읽을 수 있도록 REST Catalog에 호환 메타데이터를 등록하는 방식입니다.

아래 셀은 `iceberg_lake.bronze.*` 경로로 Iceberg metadata table을 확인합니다. 여기서 보는 `operation`은 Paimon의 `commit_kind`와 1:1로 대응하지 않습니다. 예를 들어 Paimon `$snapshots`에서 `COMPACT`가 보여도, Iceberg metadata table에서는 같은 시점이 `operation = append`로 보일 수 있습니다.

정리하면, **Paimon compaction은 Paimon `$snapshots` / `$files`로 보고, Iceberg snapshots는 reader 호환 metadata와 time travel 기준점 확인용으로 봅니다.**



In [ ]:
# Iceberg REST Catalog에 등록된 bronze namespace/table 확인
spark.sql("SHOW NAMESPACES IN iceberg_lake").show(truncate=False)
spark.sql("SHOW TABLES IN iceberg_lake.bronze").show(truncate=False)

# Iceberg metadata table: committed_at / snapshot_id / operation으로 reader metadata commit 이력을 봅니다.
# 주의: operation은 Paimon commit_kind와 1:1로 대응하지 않습니다. Paimon COMPACT도 여기서는 append로 보일 수 있습니다.
# 테이블이 아직 Iceberg-compatible metadata로 보이지 않으면 이 셀은 실패할 수 있습니다.
try:
    spark.sql("""
    SELECT
      committed_at,
      snapshot_id,
      operation
    FROM iceberg_lake.bronze.review_current.snapshots
    ORDER BY committed_at DESC
    LIMIT 5
    """).show(truncate=False)
except Exception as e:
    print("Iceberg snapshot metadata 조회 실패: 먼저 Paimon commit/compaction/metadata 생성 상태를 확인하세요.")
    print(type(e).__name__, str(e)[:500])




## 4. Paimon files: compaction과 level 분포

`row_count`는 결과를 보여주고, `snapshots`는 commit 이력을 보여주고, `files`는 내부 저장 상태를 보여줍니다. `review_current`, `order_current` 같은 primary key table은 upsert를 위해 내부적으로 level 파일을 만들고 compaction으로 정리합니다.



In [ ]:
spark.sql("""
SELECT
  level,
  COUNT(*) AS file_count,
  SUM(record_count) AS records,
  ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) AS size_mb,
  ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) AS avg_file_mb
FROM paimon_lake.bronze.`review_current$files`
GROUP BY level
ORDER BY level
""").show(truncate=False)

spark.sql("""
SELECT
  bucket,
  level,
  COUNT(*) AS files,
  SUM(record_count) AS records,
  ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) AS size_mb
FROM paimon_lake.bronze.`review_current$files`
GROUP BY bucket, level
ORDER BY bucket, level
""").show(truncate=False)

